# Why The Teacher Models Are Missing 0.90 Entity F1

This notebook focuses on one question:

**What exact error patterns are keeping the vanilla teacher and `teacher_crf` around ~0.85 test entity F1 instead of ~0.90?**

It does that in three layers:

- compares saved validation/test metrics against the `0.90` target
- summarizes the strict span confusion matrices already saved in each `summary.json`
- classifies the saved `predictions.json` into concrete error buckets like missed spans, spurious spans, and boundary overlaps

The core analysis runs from saved artifacts only. Raw token text for human-readable examples is optional and loaded only if the FiNER-ORD test split is available locally.

In [5]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from seqeval.metrics.sequence_labeling import get_entities


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data.py').exists() and (candidate / 'results').exists():
            return candidate
    raise RuntimeError('Could not find repo root containing src/data.py and results/')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RESULTS_DIR = REPO_ROOT / 'results'
TARGET_F1 = 0.90
RUN_FAMILIES = {
    'vanilla': [
        'phase_b_teacher_seed88',
        'phase_b_teacher_seed5768',
        'phase_b_teacher_seed78516',
    ],
    'crf': [
        'teacher_crf_seed88',
        'teacher_crf_seed5768',
        'teacher_crf_seed78516',
    ],
}

RESULTS_DIR

PosixPath('/Users/dayanbattulga/Desktop/personal-code/misc/dunedain_ner/results')

In [6]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text())


def load_seed_summary(run_id: str) -> dict:
    return load_json(RESULTS_DIR / run_id / 'summary.json')


def load_prediction_payload(run_id: str) -> dict:
    return load_json(RESULTS_DIR / run_id / 'predictions.json')


seed_rows = []
for family, run_ids in RUN_FAMILIES.items():
    for run_id in run_ids:
        summary = load_seed_summary(run_id)
        test_metrics = summary['test_metrics']
        seed_rows.append(
            {
                'family': family,
                'run_id': run_id,
                'seed': summary['config']['resolved_seed'],
                'best_validation_entity_f1': summary['best_validation_entity_f1'],
                'test_entity_f1': test_metrics['entity_overall_f1'],
                'gap_to_0_90': TARGET_F1 - test_metrics['entity_overall_f1'],
                'token_weighted_f1': test_metrics['token_weighted_f1'],
                'test_per_f1': test_metrics['entity_per_class'].get('PER', 0.0),
                'test_loc_f1': test_metrics['entity_per_class'].get('LOC', 0.0),
                'test_org_f1': test_metrics['entity_per_class'].get('ORG', 0.0),
                'train_time_min': summary['train_time_min'],
            }
        )

seed_df = pd.DataFrame(seed_rows).sort_values(['family', 'seed']).reset_index(drop=True)
display(seed_df.round(4))

family_df = (
    seed_df.groupby('family', as_index=False)
    .agg(
        runs=('run_id', 'count'),
        best_validation_entity_f1_mean=('best_validation_entity_f1', 'mean'),
        test_entity_f1_mean=('test_entity_f1', 'mean'),
        test_entity_f1_std=('test_entity_f1', 'std'),
        token_weighted_f1_mean=('token_weighted_f1', 'mean'),
        test_per_f1_mean=('test_per_f1', 'mean'),
        test_loc_f1_mean=('test_loc_f1', 'mean'),
        test_org_f1_mean=('test_org_f1', 'mean'),
        train_time_min_mean=('train_time_min', 'mean'),
    )
)
family_df['gap_to_0_90'] = TARGET_F1 - family_df['test_entity_f1_mean']
family_df['val_test_gap'] = (
    family_df['best_validation_entity_f1_mean'] - family_df['test_entity_f1_mean']
)
family_df['token_vs_entity_gap'] = (
    family_df['token_weighted_f1_mean'] - family_df['test_entity_f1_mean']
)
display(family_df.round(4))

notes = []
for row in family_df.itertuples(index=False):
    notes.append(
        f"- **{row.family}** averages `{row.test_entity_f1_mean:.4f}` test entity F1, which is `{row.gap_to_0_90:.4f}` below `0.90`. "
        f"Its mean best validation F1 is `{row.best_validation_entity_f1_mean:.4f}`, so the validation-to-test drop is about `{row.val_test_gap:.4f}`. "
        f"Token weighted F1 is still around `{row.token_weighted_f1_mean:.4f}`, which means the bigger problem is strict entity-span quality rather than raw token tagging."
    )

display(Markdown('### Quick read\n' + '\n'.join(notes)))


,family,run_id,seed,best_validation_entity_f1,test_entity_f1,gap_to_0_90,token_weighted_f1,test_per_f1,test_loc_f1,test_org_f1,train_time_min
0,crf,teacher_crf_seed88,88,0.9219,0.8540,0.0460,0.9853,0.9617,0.8531,0.8020,24.3906
1,crf,teacher_crf_seed5768,5768,0.9204,0.8505,0.0495,0.9855,0.9481,0.8544,0.7997,20.1636
2,crf,teacher_crf_seed78516,78516,0.9167,0.8519,0.0481,0.9852,0.9496,0.8454,0.8078,22.9008
3,vanilla,phase_b_teacher_seed88,88,0.9148,0.8468,0.0532,0.9853,0.9431,0.8627,0.7937,40.8893
4,vanilla,phase_b_teacher_seed5768,5768,0.9178,0.8510,0.0490,0.9852,0.9562,0.8444,0.8021,47.7155
5,vanilla,phase_b_teacher_seed78516,78516,0.9129,0.8475,0.0525,0.9850,0.9561,0.8544,0.7903,33.8729


,family,runs,best_validation_entity_f1_mean,test_entity_f1_mean,test_entity_f1_std,token_weighted_f1_mean,test_per_f1_mean,test_loc_f1_mean,test_org_f1_mean,train_time_min_mean,gap_to_0_90,val_test_gap,token_vs_entity_gap
0,crf,3,0.9197,0.8521,0.0018,0.9853,0.9531,0.8510,0.8032,22.4850,0.0479,0.0676,0.1332
1,vanilla,3,0.9152,0.8485,0.0022,0.9852,0.9518,0.8538,0.7954,40.8259,0.0515,0.0667,0.1367


### Quick read
- **crf** averages `0.8521` test entity F1, which is `0.0479` below `0.90`. Its mean best validation F1 is `0.9197`, so the validation-to-test drop is about `0.0676`. Token weighted F1 is still around `0.9853`, which means the bigger problem is strict entity-span quality rather than raw token tagging.
- **vanilla** averages `0.8485` test entity F1, which is `0.0515` below `0.90`. Its mean best validation F1 is `0.9152`, so the validation-to-test drop is about `0.0667`. Token weighted F1 is still around `0.9852`, which means the bigger problem is strict entity-span quality rather than raw token tagging.

## Strict Span Confusion

Each saved `summary.json` already contains a strict span confusion matrix. Those matrices are useful because they separate:

- correct spans
- missed gold spans
- spurious predicted spans
- exact-span wrong-type confusions

If the main issue were label typing, we would expect large wrong-type counts. If the main issue is span quality, we should see the error mass concentrate in missed and spurious spans instead.

In [7]:
def nested_confusion_to_frame(nested: dict) -> pd.DataFrame:
    return pd.DataFrame(nested).sort_index().sort_index(axis=1)


def mean_span_confusion_for_family(family: str) -> pd.DataFrame:
    frames = []
    for run_id in RUN_FAMILIES[family]:
        summary = load_seed_summary(run_id)
        frames.append(
            nested_confusion_to_frame(summary['test_metrics']['entity_span_confusion_matrix']).astype(float)
        )

    total = frames[0].copy()
    for frame in frames[1:]:
        total = total.add(frame, fill_value=0.0)
    return total / len(frames)


vanilla_span_cm = mean_span_confusion_for_family('vanilla')
crf_span_cm = mean_span_confusion_for_family('crf')

print('Vanilla mean strict span confusion matrix')
display(vanilla_span_cm.round(2))

print('CRF mean strict span confusion matrix')
display(crf_span_cm.round(2))

error_burden_rows = []
for family, frame in [('vanilla', vanilla_span_cm), ('crf', crf_span_cm)]:
    for entity_type in ['PER', 'LOC', 'ORG']:
        wrong_type_mean = (
            frame.loc[entity_type, ['PER', 'LOC', 'ORG']].sum()
            - frame.loc[entity_type, entity_type]
        )
        missed_mean = frame.loc[entity_type, 'Spurious_or_Missed']
        spurious_mean = frame.loc['Spurious_or_Missed', entity_type]
        error_burden_rows.append(
            {
                'family': family,
                'entity_type': entity_type,
                'correct_mean': frame.loc[entity_type, entity_type],
                'missed_mean': missed_mean,
                'spurious_mean': spurious_mean,
                'wrong_type_mean': wrong_type_mean,
                'all_error_mean': missed_mean + spurious_mean + wrong_type_mean,
            }
        )

error_burden_df = (
    pd.DataFrame(error_burden_rows)
    .sort_values(['family', 'all_error_mean'], ascending=[True, False])
    .reset_index(drop=True)
)
display(error_burden_df.round(2))

display(
    Markdown(
        '### What the span confusion says\n'
        '- The largest remaining error mass sits in **missed** and **spurious** entity spans, not same-span type swaps.\n'
        '- **ORG** is the biggest bottleneck for both teachers.\n'
        '- **LOC** is the next most fragile class.\n'
        '- **PER** is already relatively strong, so it is not the main reason the models are below `0.90`.'
    )
)


Vanilla mean strict span confusion matrix


,LOC,ORG,PER,Spurious_or_Missed
LOC,263.0,7.00,0.00,30.00
ORG,1.0,464.33,0.33,87.33
PER,3.0,6.00,270.00,7.00
Spurious_or_Missed,49.0,137.33,11.00,0.00


CRF mean strict span confusion matrix


,LOC,ORG,PER,Spurious_or_Missed
LOC,268.33,5.33,0.33,26.00
ORG,2.00,470.67,0.00,80.33
PER,3.00,1.67,274.33,7.00
Spurious_or_Missed,57.33,141.33,15.00,0.00


,family,entity_type,correct_mean,missed_mean,spurious_mean,wrong_type_mean,all_error_mean
0,crf,ORG,470.67,80.33,141.33,2.00,223.67
1,crf,LOC,268.33,26.00,57.33,5.67,89.00
2,crf,PER,274.33,7.00,15.00,4.67,26.67
3,vanilla,ORG,464.33,87.33,137.33,1.33,226.00
4,vanilla,LOC,263.00,30.00,49.00,7.00,86.00
5,vanilla,PER,270.00,7.00,11.00,9.00,27.00


### What the span confusion says
- The largest remaining error mass sits in **missed** and **spurious** entity spans, not same-span type swaps.
- **ORG** is the biggest bottleneck for both teachers.
- **LOC** is the next most fragile class.
- **PER** is already relatively strong, so it is not the main reason the models are below `0.90`.

## Saved-Prediction Error Buckets

The next section works straight from each saved `predictions.json`. It groups errors into four concrete buckets:

- `wrong_type_same_span`: exact same boundary, wrong entity type
- `boundary_or_partial_overlap`: overlapping gold and predicted spans, but not an exact strict match
- `missed_no_overlap`: a gold span with no overlapping prediction at all
- `spurious_no_overlap`: a predicted span with no overlapping gold at all

Those buckets make the bottleneck much easier to explain than one scalar F1 score.

In [8]:
def try_load_test_tokens():
    try:
        from src.data import load_finer_ord

        test_dataset = load_finer_ord()['test']
        print(f'Loaded raw test sentences: {len(test_dataset)} rows')
        return [row['gold_token'] for row in test_dataset]
    except Exception as exc:
        print('Raw token text unavailable. Continuing with span indices only.')
        print(repr(exc))
        return None


test_tokens = try_load_test_tokens()


def span_overlaps(a_start: int, a_end: int, b_start: int, b_end: int) -> bool:
    return not (a_start > b_end or b_start > a_end)


def render_span(tokens, start: int, end: int) -> str:
    if tokens is None:
        return f'[{start}:{end}]'
    return ' '.join(tokens[start : end + 1])


def classify_run_errors(family: str, run_id: str, tokens_by_sentence=None) -> list[dict]:
    payload = load_prediction_payload(run_id)
    true_labels = payload['true_labels']
    predictions = payload['predictions']
    rows = []

    for sent_idx, (gold_seq, pred_seq) in enumerate(zip(true_labels, predictions)):
        tokens = None
        if tokens_by_sentence is not None and sent_idx < len(tokens_by_sentence):
            tokens = tokens_by_sentence[sent_idx]

        sentence_text = ' '.join(tokens) if tokens is not None else None
        gold_ents = [(etype, start, end) for etype, start, end in get_entities(gold_seq)]
        pred_ents = [(etype, start, end) for etype, start, end in get_entities(pred_seq)]
        exact_pred_matches = set()

        for gold_type, gold_start, gold_end in gold_ents:
            exact_matches = [
                (pred_type, pred_start, pred_end)
                for pred_type, pred_start, pred_end in pred_ents
                if pred_start == gold_start and pred_end == gold_end
            ]

            if exact_matches:
                pred_type, pred_start, pred_end = exact_matches[0]
                exact_pred_matches.add((pred_type, pred_start, pred_end))
                if pred_type != gold_type:
                    rows.append(
                        {
                            'family': family,
                            'run_id': run_id,
                            'sent_idx': sent_idx,
                            'error_type': 'wrong_type_same_span',
                            'gold_type': gold_type,
                            'pred_type': pred_type,
                            'gold_start': gold_start,
                            'gold_end': gold_end,
                            'pred_start': pred_start,
                            'pred_end': pred_end,
                            'gold_span': render_span(tokens, gold_start, gold_end),
                            'pred_span': render_span(tokens, pred_start, pred_end),
                            'sentence': sentence_text,
                        }
                    )
                continue

            overlapping_preds = [
                (pred_type, pred_start, pred_end)
                for pred_type, pred_start, pred_end in pred_ents
                if span_overlaps(gold_start, gold_end, pred_start, pred_end)
            ]

            if overlapping_preds:
                pred_type, pred_start, pred_end = overlapping_preds[0]
                rows.append(
                    {
                        'family': family,
                        'run_id': run_id,
                        'sent_idx': sent_idx,
                        'error_type': 'boundary_or_partial_overlap',
                        'gold_type': gold_type,
                        'pred_type': pred_type,
                        'gold_start': gold_start,
                        'gold_end': gold_end,
                        'pred_start': pred_start,
                        'pred_end': pred_end,
                        'gold_span': render_span(tokens, gold_start, gold_end),
                        'pred_span': render_span(tokens, pred_start, pred_end),
                        'sentence': sentence_text,
                    }
                )
            else:
                rows.append(
                    {
                        'family': family,
                        'run_id': run_id,
                        'sent_idx': sent_idx,
                        'error_type': 'missed_no_overlap',
                        'gold_type': gold_type,
                        'pred_type': None,
                        'gold_start': gold_start,
                        'gold_end': gold_end,
                        'pred_start': None,
                        'pred_end': None,
                        'gold_span': render_span(tokens, gold_start, gold_end),
                        'pred_span': None,
                        'sentence': sentence_text,
                    }
                )

        for pred_type, pred_start, pred_end in pred_ents:
            if (pred_type, pred_start, pred_end) in exact_pred_matches:
                continue

            overlapping_gold = [
                (gold_type, gold_start, gold_end)
                for gold_type, gold_start, gold_end in gold_ents
                if span_overlaps(pred_start, pred_end, gold_start, gold_end)
            ]

            if not overlapping_gold:
                rows.append(
                    {
                        'family': family,
                        'run_id': run_id,
                        'sent_idx': sent_idx,
                        'error_type': 'spurious_no_overlap',
                        'gold_type': None,
                        'pred_type': pred_type,
                        'gold_start': None,
                        'gold_end': None,
                        'pred_start': pred_start,
                        'pred_end': pred_end,
                        'gold_span': None,
                        'pred_span': render_span(tokens, pred_start, pred_end),
                        'sentence': sentence_text,
                    }
                )

    return rows


error_rows = []
for family, run_ids in RUN_FAMILIES.items():
    for run_id in run_ids:
        error_rows.extend(classify_run_errors(family, run_id, tokens_by_sentence=test_tokens))

error_df = pd.DataFrame(error_rows)
display(error_df.head())
print(f'Total classified errors: {len(error_df)}')


Loaded raw test sentences: 1075 rows


,family,run_id,sent_idx,error_type,gold_type,pred_type,gold_start,gold_end,pred_start,pred_end,gold_span,pred_span,sentence
0,vanilla,phase_b_teacher_seed88,8,wrong_type_same_span,LOC,ORG,33.0,34.0,33.0,34.0,London Overground,London Overground,Last week a man named Robin Lee was told he wo...
1,vanilla,phase_b_teacher_seed88,11,spurious_no_overlap,NaN,LOC,NaN,NaN,14.0,14.0,NaN,Broadway,"The week before , 19 - year - old Nick Silvest..."
2,vanilla,phase_b_teacher_seed88,23,missed_no_overlap,PER,NaN,9.0,9.0,NaN,NaN,Buddha,NaN,Take it from an expert : I ’ve reached Buddha ...
3,vanilla,phase_b_teacher_seed88,41,spurious_no_overlap,NaN,PER,NaN,NaN,45.0,46.0,NaN,50 Cent,"Sure , your battery is dead , but this is why ..."
4,vanilla,phase_b_teacher_seed88,50,spurious_no_overlap,NaN,PER,NaN,NaN,13.0,13.0,NaN,Prince,The two hours when your phone died were probab...


Total classified errors: 1526


In [11]:
dominant_error_df = (
    error_df.groupby(['family', 'error_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['family', 'count'], ascending=[True, False])
)
dominant_error_df['share_within_family'] = dominant_error_df.groupby('family')['count'].transform(
    lambda values: values / values.sum()
)
display(dominant_error_df.round(4))

entity_error_df = error_df.assign(
    entity_type=error_df['gold_type'].fillna(error_df['pred_type'])
)
entity_error_df = (
    entity_error_df.groupby(['family', 'entity_type', 'error_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['family', 'count'], ascending=[True, False])
)
display(entity_error_df.head(24))

display(
    Markdown(
        '### How to read these counts\n'
        '- If `wrong_type_same_span` is small, the main issue is **not** mostly entity typing.\n'
        '- Large `boundary_or_partial_overlap` counts mean the model often finds the right neighborhood but misses exact seqeval boundaries.\n'
        '- Large `missed_no_overlap` and `spurious_no_overlap` counts mean recall and precision are both leaking on whole spans, especially in harder classes.'
    )
)


,family,error_type,count,share_within_family
2,crf,spurious_no_overlap,381,0.5026
0,crf,boundary_or_partial_overlap,199,0.2625
1,crf,missed_no_overlap,141,0.1860
3,crf,wrong_type_same_span,37,0.0488
6,vanilla,spurious_no_overlap,343,0.4466
4,vanilla,boundary_or_partial_overlap,189,0.2461
5,vanilla,missed_no_overlap,184,0.2396
7,vanilla,wrong_type_same_span,52,0.0677


,family,entity_type,error_type,count
6,crf,ORG,spurious_no_overlap,287
4,crf,ORG,boundary_or_partial_overlap,121
5,crf,ORG,missed_no_overlap,120
2,crf,LOC,spurious_no_overlap,74
0,crf,LOC,boundary_or_partial_overlap,65
10,crf,PER,spurious_no_overlap,20
3,crf,LOC,wrong_type_same_span,17
11,crf,PER,wrong_type_same_span,14
1,crf,LOC,missed_no_overlap,13
8,crf,PER,boundary_or_partial_overlap,13


### How to read these counts
- If `wrong_type_same_span` is small, the main issue is **not** mostly entity typing.
- Large `boundary_or_partial_overlap` counts mean the model often finds the right neighborhood but misses exact seqeval boundaries.
- Large `missed_no_overlap` and `spurious_no_overlap` counts mean recall and precision are both leaking on whole spans, especially in harder classes.

In [12]:
example_df = error_df.drop_duplicates(
    subset=['family', 'sent_idx', 'error_type', 'gold_type', 'pred_type', 'gold_span', 'pred_span']
).reset_index(drop=True)


def show_examples(family: str, error_type: str, entity_type: str | None = None, n: int = 5) -> pd.DataFrame:
    subset = example_df[(example_df['family'] == family) & (example_df['error_type'] == error_type)]
    if entity_type is not None:
        subset = subset[
            (subset['gold_type'] == entity_type) | (subset['pred_type'] == entity_type)
        ]

    cols = [
        'run_id',
        'sent_idx',
        'error_type',
        'gold_type',
        'pred_type',
        'gold_start',
        'gold_end',
        'pred_start',
        'pred_end',
        'gold_span',
        'pred_span',
    ]
    if subset['sentence'].notna().any():
        cols.append('sentence')
    return subset[cols].head(n)


print('Vanilla ORG missed spans')
display(show_examples('vanilla', 'missed_no_overlap', 'ORG', n=5))

print('Vanilla ORG boundary / partial-overlap errors')
display(show_examples('vanilla', 'boundary_or_partial_overlap', 'ORG', n=5))

print('CRF ORG spurious spans')
display(show_examples('crf', 'spurious_no_overlap', 'ORG', n=5))

print('CRF LOC boundary / partial-overlap errors')
display(show_examples('crf', 'boundary_or_partial_overlap', 'LOC', n=5))


Vanilla ORG missed spans


,run_id,sent_idx,error_type,gold_type,pred_type,gold_start,gold_end,pred_start,pred_end,gold_span,pred_span,sentence
13,phase_b_teacher_seed88,74,missed_no_overlap,ORG,NaN,2.0,2.0,NaN,NaN,TMFMarlowe,NaN,"John Rosevear TMFMarlowe Jul 20 , 2015 at 7:07..."
21,phase_b_teacher_seed88,117,missed_no_overlap,ORG,NaN,1.0,1.0,NaN,NaN,Economist,NaN,"The Economist is calling it "" transformative ""..."
22,phase_b_teacher_seed88,123,missed_no_overlap,ORG,NaN,1.0,1.0,NaN,NaN,Fools,NaN,"We Fools may not all hold the same opinions , ..."
32,phase_b_teacher_seed88,158,missed_no_overlap,ORG,NaN,17.0,17.0,NaN,NaN,Facebook,NaN,For more information about our facilities and ...
35,phase_b_teacher_seed88,173,missed_no_overlap,ORG,NaN,4.0,5.0,NaN,NaN,BUSINESS WIRE,NaN,"DULUTH , Ga .--( BUSINESS WIRE )-- AGCO , Your..."


Vanilla ORG boundary / partial-overlap errors


,run_id,sent_idx,error_type,gold_type,pred_type,gold_start,gold_end,pred_start,pred_end,gold_span,pred_span,sentence
5,phase_b_teacher_seed88,54,boundary_or_partial_overlap,ORG,ORG,3.0,4.0,3.0,3.0,NSW government,NSW,That 's the NSW government 's vision for Austr...
10,phase_b_teacher_seed88,65,boundary_or_partial_overlap,ORG,ORG,44.0,45.0,44.0,44.0,AAP .,AAP,""" I think it 's very important that we do this..."
12,phase_b_teacher_seed88,70,boundary_or_partial_overlap,ORG,ORG,9.0,10.0,9.0,9.0,Facebook Sydney,Facebook,"@ YahooNZBusiness on Twitter , become a fan on..."
23,phase_b_teacher_seed88,140,boundary_or_partial_overlap,ORG,ORG,8.0,12.0,8.0,11.0,"Commodity Systems , Inc .","Commodity Systems , Inc",Historical chart data and daily updates provid...
24,phase_b_teacher_seed88,141,boundary_or_partial_overlap,ORG,ORG,9.0,12.0,9.0,11.0,"Morningstar , Inc .","Morningstar , Inc",International historical chart data and daily ...


CRF ORG spurious spans


,run_id,sent_idx,error_type,gold_type,pred_type,gold_start,gold_end,pred_start,pred_end,gold_span,pred_span,sentence
349,teacher_crf_seed88,74,spurious_no_overlap,NaN,ORG,NaN,NaN,15.0,15.0,NaN,Buick,"John Rosevear TMFMarlowe Jul 20 , 2015 at 7:07..."
351,teacher_crf_seed88,93,spurious_no_overlap,NaN,ORG,NaN,NaN,12.0,12.0,NaN,Buick,The Envision is a midsize crossover SUV that s...
352,teacher_crf_seed88,94,spurious_no_overlap,NaN,ORG,NaN,NaN,14.0,14.0,NaN,Buick,"Both of those are also sold in China , and dem..."
353,teacher_crf_seed88,98,spurious_no_overlap,NaN,ORG,NaN,NaN,0.0,0.0,NaN,Ford,Ford ( NYSE : F ) has posted huge growth in Ch...
359,teacher_crf_seed88,142,spurious_no_overlap,NaN,ORG,NaN,NaN,3.0,4.0,NaN,News Network,Yahoo ! - News Network


CRF LOC boundary / partial-overlap errors


,run_id,sent_idx,error_type,gold_type,pred_type,gold_start,gold_end,pred_start,pred_end,gold_span,pred_span,sentence
341,teacher_crf_seed88,56,boundary_or_partial_overlap,LOC,LOC,19.0,25.0,24.0,25.0,inner - west - spanning Glebe Island,Glebe Island,However wires are crossed on how exactly Mr Ba...
343,teacher_crf_seed88,58,boundary_or_partial_overlap,LOC,ORG,17.0,19.0,17.0,18.0,Australian Technology Park,Australian Technology,The founders of Australian software giant Atla...
344,teacher_crf_seed88,59,boundary_or_partial_overlap,LOC,LOC,20.0,21.0,20.0,20.0,Redfern centre,Redfern,The country 's most successful startup said it...
361,teacher_crf_seed88,144,boundary_or_partial_overlap,ORG,LOC,19.0,22.0,19.0,19.0,Japan Keio Plaza Hotel,Japan,""" The Joy of Sports Photography 2015 "" Co-spon..."
362,teacher_crf_seed88,144,boundary_or_partial_overlap,LOC,LOC,29.0,31.0,29.0,29.0,"Shinjuku , Tokyo",Shinjuku,""" The Joy of Sports Photography 2015 "" Co-spon..."


## What This Notebook Should Show

On the current saved runs, the story should be pretty consistent:

- both teachers are still about five F1 points short of `0.90` on **test**
- both have a much stronger **validation** score than **test** score, so the problem is not just fitting the training distribution
- token-level weighted F1 stays near `0.985`, which means the strict seqeval entity metric is being lost mostly on exact span quality
- the dominant failures are **missed** and **spurious** spans, not exact same-span type swaps
- **ORG** is the biggest remaining bottleneck and **LOC** is next
- CRF helps a little on ORG, but it does not remove the broader span-boundary problem

That is the clearest explanation for why these teachers are still below `0.90` entity F1.